# **1. Perkenalan Dataset**

Dataset yang digunakan adalah **Breast Cancer Wisconsin Dataset**.

**Deskripsi:**
- **Sumber:** Scikit-learn built-in datasets (awalnya dari UCI ML Repository)
- **Jumlah sampel:** 569 baris
- **Jumlah fitur:** 30 fitur numerik (hasil pengukuran sel tumor)
- **Target:** Diagnosis (0=Malignant/Ganas, 1=Benign/Jinak)
- **Task:** Klasifikasi biner

Dataset ini dipilih karena:
1. Belum digunakan dalam modul pembelajaran Dicoding
2. Memiliki fitur yang beragam dan cocok untuk demonstrasi pipeline ML
3. Relevan dengan dunia nyata (deteksi kanker)


# **2. Import Library**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

print('Semua library berhasil diimport!')
print(f'Pandas  : {pd.__version__}')
print(f'NumPy   : {np.__version__}')

# **3. Memuat Dataset**

In [ ]:
# Load dataset
bc = load_breast_cancer()
df = pd.DataFrame(data=bc.data, columns=bc.feature_names)
df['target'] = bc.target
df['diagnosis'] = df['target'].map({0: 'malignant', 1: 'benign'})

print('Dataset berhasil dimuat!')
print(f'Shape: {df.shape}')
print(f'Fitur: {list(bc.feature_names)}')
df.head()

In [ ]:
# Simpan raw dataset
os.makedirs('../breast_cancer_raw', exist_ok=True)
df.to_csv('../breast_cancer_raw/breast_cancer_raw.csv', index=False)
print('Raw dataset disimpan ke ../breast_cancer_raw/breast_cancer_raw.csv')

# **4. Exploratory Data Analysis (EDA)**
Melakukan eksplorasi mendalam untuk memahami karakteristik dataset.

In [ ]:
# 4.1 Informasi Dasar
print('='*55)
print('INFORMASI DASAR DATASET')
print('='*55)
print(f'Jumlah baris  : {df.shape[0]}')
print(f'Jumlah kolom  : {df.shape[1]}')
df.info()

In [ ]:
# 4.2 Statistik Deskriptif
print('STATISTIK DESKRIPTIF')
df.describe().round(3)

In [ ]:
# 4.3 Missing Values
print('MISSING VALUES')
print(df.isnull().sum())
print(f'Total missing: {df.isnull().sum().sum()}')

In [ ]:
# 4.4 Duplikasi
print(f'Jumlah duplikasi: {df.duplicated().sum()}')

In [ ]:
# 4.5 Distribusi Target
print('DISTRIBUSI TARGET')
print(df['diagnosis'].value_counts())

plt.figure(figsize=(6,4))
df['diagnosis'].value_counts().plot(kind='bar', color=['salmon','steelblue'])
plt.title('Distribusi Kelas Target')
plt.xlabel('Diagnosis')
plt.ylabel('Jumlah')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# 4.6 Distribusi 5 Fitur Utama
top_features = list(bc.feature_names[:5])
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, col in enumerate(top_features):
    for diag, color in zip(['malignant','benign'], ['salmon','steelblue']):
        axes[i].hist(df[df['diagnosis']==diag][col], alpha=0.6,
                     label=diag, color=color, bins=20)
    axes[i].set_title(col[:15])
    axes[i].legend(fontsize=7)
plt.suptitle('Distribusi 5 Fitur Utama per Diagnosis', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 4.7 Correlation Heatmap (10 fitur pertama)
plt.figure(figsize=(10, 8))
corr_cols = list(bc.feature_names[:10]) + ['target']
corr = df[corr_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Heatmap (10 Fitur Pertama)')
plt.tight_layout()
plt.show()

In [ ]:
# 4.8 Boxplot Deteksi Outlier
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for i, col in enumerate(top_features):
    df.boxplot(column=col, by='diagnosis', ax=axes[i])
    axes[i].set_title(col[:15])
plt.suptitle('Boxplot - Deteksi Outlier per Diagnosis')
plt.tight_layout()
plt.show()

**Ringkasan EDA:**
- Dataset memiliki **569 sampel** dengan **30 fitur numerik**
- Tidak ada missing values dan tidak ada duplikasi
- Distribusi kelas: 212 malignant (37.3%) dan 357 benign (62.7%) — sedikit imbalanced
- Fitur seperti `mean radius`, `mean perimeter` memiliki korelasi tinggi dengan target
- Terdapat outlier pada beberapa fitur, perlu ditangani
- Skala fitur berbeda-beda, perlu normalisasi

# **5. Data Preprocessing**

Tahapan preprocessing:
1. Hapus duplikasi
2. Tangani missing values
3. Deteksi dan tangani outlier (IQR)
4. Normalisasi fitur (StandardScaler)
5. Split data (train/test)
6. Simpan dataset hasil preprocessing

In [ ]:
# 5.1 Hapus Duplikasi
df_clean = df.copy()
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f'Duplikat dihapus: {before - len(df_clean)} baris')
print(f'Shape setelah: {df_clean.shape}')

In [ ]:
# 5.2 Tangani Missing Values
feature_cols = list(bc.feature_names)
print('Missing values sebelum handling:')
print(df_clean[feature_cols].isnull().sum().sum())
for col in feature_cols:
    if df_clean[col].isnull().any():
        df_clean[col].fillna(df_clean[col].median(), inplace=True)
print(f'Missing values setelah handling: {df_clean[feature_cols].isnull().sum().sum()}')

In [ ]:
# 5.3 Tangani Outlier (IQR)
def remove_outliers_iqr(df, columns):
    df_out = df.copy()
    total = 0
    for col in columns:
        Q1 = df_out[col].quantile(0.25)
        Q3 = df_out[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        before = len(df_out)
        df_out = df_out[(df_out[col] >= lower) & (df_out[col] <= upper)]
        removed = before - len(df_out)
        total += removed
        if removed > 0:
            print(f'{col}: {removed} outlier dihapus')
    print(f'Total outlier dihapus: {total}')
    print(f'Shape setelah: {df_out.shape}')
    return df_out

df_clean = remove_outliers_iqr(df_clean, feature_cols)

In [ ]:
# 5.4 Normalisasi Fitur
X = df_clean[feature_cols].copy()
y = df_clean['target'].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols, index=X.index)

print('Statistik sebelum normalisasi:')
print(X.describe().round(3).iloc[:, :5])
print('\nStatistik setelah normalisasi:')
print(X_scaled_df.describe().round(3).iloc[:, :5])

In [ ]:
# 5.5 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Total data  : {len(X_scaled_df)}')
print(f'Data train  : {len(X_train)} ({len(X_train)/len(X_scaled_df)*100:.1f}%)')
print(f'Data test   : {len(X_test)} ({len(X_test)/len(X_scaled_df)*100:.1f}%)')
print(f'\nDistribusi train:\n{y_train.value_counts()}')
print(f'\nDistribusi test:\n{y_test.value_counts()}')

In [ ]:
# 5.6 Simpan Dataset Hasil Preprocessing
os.makedirs('../breast_cancer_preprocessing', exist_ok=True)

train_df = X_train.copy()
train_df['target'] = y_train.values
test_df = X_test.copy()
test_df['target'] = y_test.values

train_df.to_csv('../breast_cancer_preprocessing/breast_cancer_train.csv', index=False)
test_df.to_csv('../breast_cancer_preprocessing/breast_cancer_test.csv', index=False)

print('Dataset preprocessing berhasil disimpan!')
print(f'  - breast_cancer_train.csv : {train_df.shape}')
print(f'  - breast_cancer_test.csv  : {test_df.shape}')

**Ringkasan Preprocessing:**
- Tidak ada duplikasi yang dihapus
- Tidak ada missing values
- Outlier ditangani menggunakan metode IQR
- Fitur dinormalisasi menggunakan StandardScaler
- Data dibagi 80% training dan 20% testing dengan stratified split
- Dataset siap latih disimpan ke folder `breast_cancer_preprocessing/`